# Nueva Córdoba en PyBullet

**Objetivo:** cargar el mapa a escala real y probar el CF2X, las colisiones y las opciones físicas. No ejecuta una DQN ni entrena un modelo.

Usá CPU. Ejecutá las celdas en orden. Necesitás el ZIP completo del repositorio.

© OpenStreetMap contributors · Overture Maps · Microsoft. Datos: ODbL y atribuciones incluidas. La mayoría de las alturas son estimaciones; el terreno es plano. Las decoraciones del visor no equivalen a un relevamiento de obstáculos.

## 1. Instalar las dependencias

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pybullet==3.2.7', 'numpy>=2,<3', 'Pillow>=10,<14'])
print('Dependencias listas.')

## 2. Subir el ZIP

Seleccioná el ZIP que descargaste de GitHub con **Code → Download ZIP**, o el ZIP del proyecto que recibiste. No subas credenciales de Kaggle ni de GitHub.

La celda reconoce una carpeta ya cargada si la ejecutás de nuevo.

In [ ]:
from pathlib import Path
import zipfile, sys, os

if (Path.cwd() / 'ciudad.json').is_file():
    carpeta_mapa = Path.cwd()
elif 'carpeta_mapa' in globals() and (Path(carpeta_mapa) / 'ciudad.json').is_file():
    carpeta_mapa = Path(carpeta_mapa)
else:
    from google.colab import files
    subidos = files.upload()
    archivo = next((Path(nombre) for nombre in subidos if nombre.lower().endswith('.zip')), None)
    if archivo is None:
        raise ValueError('Seleccioná el ZIP completo del repositorio.')
    destino = Path('/content/nueva_cordoba_equipo')
    destino.mkdir(exist_ok=True)
    with zipfile.ZipFile(archivo) as z:
        for miembro in z.infolist():
            objetivo = (destino / miembro.filename).resolve()
            if not objetivo.is_relative_to(destino.resolve()):
                raise ValueError('El ZIP contiene una ruta fuera de la carpeta de destino.')
        z.extractall(destino)
    candidatos = list(destino.rglob('ciudad_pybullet.py'))
    if len(candidatos) != 1:
        raise ValueError('El ZIP debe contener una sola carpeta del proyecto con ciudad_pybullet.py.')
    carpeta_mapa = candidatos[0].parent
sys.path.insert(0, str(carpeta_mapa))
os.chdir(carpeta_mapa)
print('Mapa encontrado:', carpeta_mapa)
assert (carpeta_mapa / 'configuracion.json').is_file()

## 3. Elegir las opciones

Se carga un sector para que la simulación sea liviana. Su centro predeterminado es el Buen Pastor; podés cambiarlo en `configuracion.json`.

El viento usa ráfagas aleatorias reproducibles y aumenta con la altura. Los autos son vehículos físicos simplificados. Árboles y sus colisiones son una distribución ilustrativa.

Primero probá todo apagado. Después activá una opción por vez para comparar.

In [ ]:
#@title Opciones del escenario
ACTIVAR_VIENTO = False #@param {type:"boolean"}
VELOCIDAD_VIENTO_M_S = 2.0 #@param {type:"number"}
RAFAGAS = 0.35 #@param {type:"number"}
AUTOS_FISICOS = False #@param {type:"boolean"}
CANTIDAD_AUTOS = 4 #@param {type:"integer"}
VELOCIDAD_AUTOS_M_S = 5.0 #@param {type:"number"}
COLISION_ARBOLES = False #@param {type:"boolean"}
RUIDO_SENSOR_M = 0.0 #@param {type:"number"}
LATENCIA_SENSOR_MS = 0.0 #@param {type:"number"}
RADIO_SECTOR_M = 160 #@param {type:"integer"}
SEMILLA = 2026 #@param {type:"integer"}

from fisica_urbana import configuracion
cfg = configuracion()
cfg['semilla'] = SEMILLA
cfg['viento']['activo'] = ACTIVAR_VIENTO
cfg['viento']['velocidad_referencia_m_s'] = VELOCIDAD_VIENTO_M_S
cfg['viento']['intensidad_rafagas'] = RAFAGAS
cfg['vehiculos'].update(activos=AUTOS_FISICOS, cantidad=CANTIDAD_AUTOS,
                        velocidad_m_s=VELOCIDAD_AUTOS_M_S)
cfg['arboles']['colisiones'] = COLISION_ARBOLES
cfg['sensores'].update(ruido_sigma_m=RUIDO_SENSOR_M, latencia_ms=LATENCIA_SENSOR_MS)
cfg['sector'] = {**cfg.get('sector', {}), 'cargar_todo': False, 'radio_m': RADIO_SECTOR_M}
assert 50 <= RADIO_SECTOR_M <= 500
assert 0 <= CANTIDAD_AUTOS <= 30
assert VELOCIDAD_VIENTO_M_S >= 0 and VELOCIDAD_AUTOS_M_S >= 0
print('Configuración lista; se aplicará en la próxima celda.')

## 4. Cargar el mapa y el dron

La prueba usa el modelo físico CF2X del proyecto: masa 27 g y cilindro de colisión de 12 cm de diámetro por 2,5 cm de alto. PyBullet añade un pequeño margen de colisión.

Se coloca sobre un techo para verificar el apoyo físico. **No es un vuelo controlado:** acá no se aplican las fuerzas de los motores ni una política DQN.

In [ ]:
import pybullet as p
import numpy as np
from ciudad_pybullet import cargar_dron, rayos
from fisica_urbana import crear_escenario

if 'cliente' in globals() and p.isConnected(cliente):
    p.disconnect(cliente)
cliente = p.connect(p.DIRECT)
p.setTimeStep(1 / 240, physicsClientId=cliente)
p.setGravity(0, 0, -9.81, physicsClientId=cliente)
mapa, mundo = crear_escenario(cliente, cfg)
assert mapa['edificios'], 'El sector no contiene construcciones.'

techo = max(mapa['edificios'], key=lambda b: b['h'])
v = np.asarray(techo['v'])
t = np.asarray(techo['tri'])
a, b, c = v[t[:, 0]], v[t[:, 1]], v[t[:, 2]]
areas = np.abs((b[:, 0]-a[:, 0])*(c[:, 1]-a[:, 1]) - (b[:, 1]-a[:, 1])*(c[:, 0]-a[:, 0]))
xy = v[t[np.argmax(areas)]].mean(axis=0)
dron = cargar_dron(cliente, [*xy, techo['h'] + 0.5])
obstaculos = mapa['ids_obstaculos'] + mundo.ids_vehiculos
masa = p.getDynamicsInfo(dron, -1, physicsClientId=cliente)[0]
assert abs(masa - 0.027) < 1e-6
print('Construcciones cargadas:', len(mapa['edificios']))
print('Autos físicos:', len(mundo.ids_vehiculos))
print('Masa del dron:', masa, 'kg')

## 5. Comprobar física y sensores

La caída prueba el contacto con el techo. Si activás viento, el dron no tiene un controlador que lo compense en esta demostración.

Al integrar la DQN, el entorno debe aplicar sus fuerzas de motores además de estas perturbaciones.

In [ ]:
for paso in range(480):
    mundo.aplicar(1 / 240, dron_id=dron)
    p.stepSimulation(physicsClientId=cliente)

posicion = p.getBasePositionAndOrientation(dron, physicsClientId=cliente)[0]
contactos = p.getContactPoints(bodyA=dron, physicsClientId=cliente)
lecturas = rayos(cliente, posicion, obstaculos, alcance=0.5)
lecturas = mundo.degradar_sensores(lecturas, alcance=0.5)
print('Posición final (m):', np.round(posicion, 3))
print('Contactos físicos:', len(contactos))
print('Sensores horizontales (m):', np.round(lecturas, 3))
print('Viento actual (m/s):', np.round(mundo.ultimo_viento, 3))
print('Fuerza de viento (N):', np.round(mundo.ultima_fuerza, 5))
assert np.isfinite(posicion).all()
assert len(lecturas) == 8 and np.isfinite(lecturas).all()
if not ACTIVAR_VIENTO:
    assert contactos, 'Revisar contacto con el techo.'
print('Prueba básica terminada. Esto no mide el rendimiento de una DQN.')

## 6. Ver el sector y el dron

In [ ]:
from PIL import Image
from IPython.display import display

def mostrar_camara(objetivo, distancia, ancho=1000, alto=650):
    vista = p.computeViewMatrixFromYawPitchRoll(objetivo, distancia, 35, -45, 0, 2)
    proyeccion = p.computeProjectionMatrixFOV(55, ancho / alto, 0.05, 2000)
    _, _, rgba, _, _ = p.getCameraImage(ancho, alto, vista, proyeccion,
        renderer=p.ER_TINY_RENDERER, physicsClientId=cliente)
    imagen = Image.fromarray(np.asarray(rgba, dtype=np.uint8).reshape(alto, ancho, 4))
    display(imagen)

centro = mapa['centro_activo']
mostrar_camara([centro[0], centro[1], 15], RADIO_SECTOR_M * 2.8)
mostrar_camara(list(posicion), 1.0, 900, 600)

## Siguiente paso del equipo: conectar el agente

Reemplazá el generador de edificios artificiales por `crear_escenario(env.CLIENT, cfg)`. Recreá mapa y opciones después de cada reinicio del simulador. Incluí los IDs de autos y árboles en sensores y choques.

Llamá a `mundo.aplicar(1/self.PYB_FREQ, dron_id)` en cada subpaso físico, después de las fuerzas de motores. No agregues un segundo `stepSimulation` por fuera de `env.step`.

El responsable del entorno debe revisar observaciones, puntos libres de inicio/destino, límites y duración antes de cargar el checkpoint. Conservar el número de entradas de una red no garantiza que funcione en esta distribución.

El visor de `index.html` ofrece una presentación visual más detallada. Las imágenes de esta notebook muestran la geometría física de PyBullet.